In [18]:
import numpy as np
import matplotlib.pyplot as plt

def plot_voxel_by_index(skel, flat_idx, cmap='gray'):
    """
    Visualize a voxel in a 3D skeleton array by its flattened index.

    Parameters
    ----------
    skel : np.ndarray
        3D array (e.g., skeleton volume).
    flat_idx : int
        Flattened index from np.flatnonzero(skel).
    cmap : str, optional
        Colormap for display (default 'gray').
    """
    # Get (y, x, z) coordinates from flat index
    y, x, z = np.unravel_index(flat_idx, skel.shape)

    # Show the correct slice
    plt.imshow(skel[:, :, z], cmap=cmap, origin='upper')

    # Highlight voxel
    plt.scatter(x, y, color='red', s=10, marker='s', label=f'Flat idx {flat_idx}')
    plt.colorbar()
    plt.title(f'z = {z}  |  voxel = ({y}, {x}, {z})')
    plt.show()

def BinarizeImage(Im:np.ndarray, frangi_sigma = 8, frangi_enabled = True, method = "Fuzzy Means") -> np.ndarray:
    """
    Binarizes an image using the selected method and options from the GUI app.

    Parameters:
    - app: GUI application object containing UI state (e.g., checkboxes, dropdowns)
    - Im: Input image as a 2D NumPy array (grayscale)

    Returns:
    - Binarized image as a 2D NumPy array (dtype=float)
    """
    # Find the brightest pixel in the image
    col_max_vals = np.max(Im, axis=0) # the maximum of each column
    cc = np.argmax(col_max_vals) #  gets the index of the column where the brightest value occurred
    rr = np.argmax(Im[:, cc]) # captures the row index of that max value

    #	•	rr: row of the brightest pixel
	#   •	cc: column of the brightest pixel
    #   •	So (rr, cc) is the coordinate of the brightest pixel in the whole image.

    # Apply Frangi filter if selected
    if frangi_enabled:
        opts = {
            'sigmarange': [1, frangi_sigma],
            'sigmastepsize': 2,
            'correctionconst1': 0.8,
            'correctionconst2': 15
        }
        Im = frangi_2Dfilter(Im, opts)

    # Apply the selected binarization method
    if method == 'Fuzzy Means':
        nth = 2
        Im = fuzzy_thresholding(Im, nth, 3) - 1

    elif method == 'Adaptive Thresholding':
        Im = adaptive_thresholding(Im, 70).astype(float)

    else:
        raise ValueError("Incorrect Segmentation Method")

    # Ensure vessels are white, background is black, turning the image into black and white
    vessel_value = Im[rr, cc]
    Im[Im == vessel_value] = 255
    Im[Im != 255] = 0

    return Im

def overlay_mask(img, mask, color=(1, 0, 0), alpha=0.5):
    """
    Overlay a binary mask on a grayscale image.

    Parameters:
    - img: grayscale image (H,W)
    - mask: binary mask (H,W), values 0 or 255
    - color: tuple with RGB values (0-1 range for matplotlib)
    - alpha: transparency of overlay

    Returns:
    - RGB image with overlay
    """
    # Normalize grayscale for display
    img_norm = cv.normalize(img, None, 0, 1, cv.NORM_MINMAX)

    # Convert grayscale to RGB
    img_rgb = np.stack([img_norm]*3, axis=-1)

    # Create color mask
    mask_bool = mask > 0
    overlay = np.zeros_like(img_rgb)
    overlay[mask_bool] = color

    # Blend images
    blended = (1 - alpha) * img_rgb + alpha * overlay
    return blended


def center_crop(img, dim):
    """
    Center crop an image (2D NumPy array).

    Parameters:
        img (np.ndarray): Input image (H x W).
        dim (tuple): Desired output size (h, w).

    Returns:
        np.ndarray: Cropped image of size (h, w).
    """
    h, w = img.shape[:2]
    new_h, new_w = dim

    if new_h > h or new_w > w:
        raise ValueError("Crop size must be smaller than the image size")

    top = (h - new_h) // 2
    left = (w - new_w) // 2

    return img[top: top + new_h, left: left + new_w]


In [19]:
from skimage.io import imread
from skimage.filters import frangi
import matplotlib.pyplot as plt
from skimage.morphology import skeletonize, remove_small_objects
from scipy.ndimage import distance_transform_edt
from Skel2Graph3D import skel2graph3d

import sys
sys.path.append("../../")
from frangi_2dfilter import frangi_2Dfilter
from fuzzy_thresholding import fuzzy_thresholding
from adaptive_thresholding import adaptive_thresholding

octa = imread("../../../assets/MIPs/10301.bmp")

# Frangi vessel enhancement
vessels = frangi(
    octa,
    scale_range=(0.5, 4),
    scale_step=0.5,
    alpha=0.5,
    beta=0.5,
    gamma=4,
    black_ridges=False
)

adpt_mask = BinarizeImage(vessels, frangi_sigma=8, frangi_enabled=False,
                            method="Adaptive Thresholding")

adpt_mask_cropped = center_crop(adpt_mask, dim=(20, 20)).astype(np.uint16)
skel = skeletonize(adpt_mask).astype(np.uint)

node, link = skel2graph3d(skel, 10)

/var/folders/j8/24ft1_090l56tpp7kwbz30gm0000gn/T/ipykernel_15334/1877419625.py:17: UserWarning: Use keyword parameter `sigmas` instead of `scale_range` and `scale_range` which will be removed in version 0.17.
  vessels = frangi(


In [20]:
len(node)

3100

In [21]:
len(link)

1361

In [22]:
print(node[2])

node_cluster(
  nodes_idx: [2374 3289 3292 4210]
  links: [2, 3]
  conn: [1222, 52]
  ep: 0
  comx: 3.0
  comy: 178.75
  comz: 1.0
  label: 2
)


In [23]:
len(node[2].links)

2

In [24]:
print(link[0])

Link(
  start: 0
  end: 26
  path: [4759, 5674, 6592, 7507, 8422, 9337]
  label: 2
)
